# `metrics` -- Calibration & Skill Metrics

This project treats **calibration** (does a predicted 70% actually happen 70% of the time?) as being just as important as raw accuracy -- arguably more important for a win-probability model, where the *number itself* is the product, not just a threshold for a yes/no decision. This module provides the two metrics used throughout `pipeline.py` to measure that: **ECE** and **Brier Skill Score**.

In [1]:
import numpy as np
from sklearn.metrics import brier_score_loss

## `ece()` -- Expected Calibration Error

**Intuition:** split all predictions into 10 equal-width bins by predicted probability (0.0-0.1, 0.1-0.2, ..., 0.9-1.0). Within each bin, compare the *average predicted probability* to the *actual observed frequency* of the positive class. A well-calibrated model has these two numbers close together in every bin.

ECE is the **weighted average absolute gap** between predicted and observed, weighted by how many predictions fall in each bin (so a bin with almost no predictions in it doesn't dominate the score). Lower is better; 0 is perfect calibration.

Bins with zero predictions are skipped entirely (there's nothing to compare).

In [2]:
def ece(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    """
    Expected Calibration Error: weighted mean absolute difference between
    predicted probability and observed frequency across equal-width bins.

    Returns a value in [0, 1]; lower is better-calibrated.
    """
    bins = np.linspace(0, 1, n_bins + 1)
    err = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (y_prob >= lo) & (y_prob < hi)
        if m.sum() == 0:
            continue
        err += m.mean() * abs(y_true[m].mean() - y_prob[m].mean())
    return float(err)

## `brier_skill_score()` -- accuracy relative to a naive baseline

The raw Brier score (mean squared error between predicted probability and the 0/1 outcome) is hard to interpret in isolation -- is 0.25 good or bad? It depends entirely on how predictable the event is in the first place.

**BSS answers a more useful question: "how much better is this model than just always guessing the historical base rate (climatology)?"**

`BSS = 1 - BS(model) / BS(climatology)`
- `BSS = 0`: no better than the naive baseline.
- `BSS = 1`: perfect predictions.
- `BSS < 0`: **worse** than just guessing the base rate -- this   is exactly what happens with project_gagan's pre-match model   (BSS around -0.02 to -0.05), an honest signal that pre-match   T20 outcomes are close to unpredictable from the available   features.

**Why `p_clim` must be passed in explicitly, not computed from the test set** (marked `DEF-M03` in the original code): the climatology baseline has to be the **training-set** base rate. If you computed it from the test set's own labels instead, you'd be leaking test-set label information into your baseline, which would make BSS look artificially better than it should.

In [3]:
def brier_skill_score(
    y: np.ndarray,
    p: np.ndarray,
    p_clim: np.ndarray,
) -> float:
    """
    Brier Skill Score relative to a reference climatology forecast.

    BSS = 1 - BS(model) / BS(climatology).
    BSS = 0 means no improvement over climatology; BSS = 1 is perfect.
    Negative values indicate the model is worse than climatology.

    DEF-M03: requires an explicit p_clim (training base-rate) to avoid
    leaking test-set label distribution into the reference forecast.
    """
    return float(1 - brier_score_loss(y, p) / brier_score_loss(y, p_clim))